# Group-aware columns: transform, rank and rolling

Python stage, lesson P20. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** add a group's total to every row with `transform`; rank rows within a group with `rank`; compare with the previous row of the same group with `shift`; compute a rolling average and see what it gives before the window is full.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# One row for each order line with its store, month and category, as built in lessons P13, P15 and P17.
price_of = dict(zip(products["product_id"], products["unit_price_usd"]))
lines = order_items.assign(price=order_items["product_id"].map(price_of))
lines = lines.assign(amount=lines["quantity"] * lines["price"] - lines["discount_usd"])
sales = lines.merge(orders[["order_id", "store_id", "ordered_at"]], on="order_id", validate="many_to_one")
sales = sales.merge(products[["product_id", "category"]], on="product_id", validate="many_to_one")
sales = sales.assign(month=pd.to_datetime(sales["ordered_at"]).dt.month)
# The sales of each store in each month (lesson P19), and the orders of each customer in each store.
store_month = sales.groupby(["store_id", "month"])["amount"].sum().reset_index()
per_customer = orders.dropna(subset=["customer_id"]).groupby(["store_id", "customer_id"]).size().reset_index(name="orders")

## Unit 1: transform: the group's answer on every row

**Predict first.** how many values will each of the two have, and what will the first three values of `broadcast` be?

In [ ]:
totals = store_month.groupby("store_id")["amount"].sum()
broadcast = store_month.groupby("store_id")["amount"].transform("sum")
print(len(totals), len(broadcast))
print(broadcast.head(3).tolist())

**Predict first.** which store and month will have the biggest share of its own store's sales, and how many of the first four rows will belong to that store?

In [ ]:
store_month["store_total"] = broadcast
store_month["share"] = store_month["amount"] / store_month["store_total"]
print(store_month.sort_values("share", ascending=False).head(4).round(3))

## Unit 2: rank: places, with ties

**Predict first.** four customers have 10 orders and three have 9. What will `rank_min` and `rank_dense` show for the customers with 9?

In [ ]:
harbor = per_customer[per_customer["store_id"] == 3]
ranked = harbor.assign(
    rank_min=harbor["orders"].rank(method="min", ascending=False),
    rank_dense=harbor["orders"].rank(method="dense", ascending=False),
)
print(ranked.sort_values(["rank_min", "customer_id"]).head(8))

**Predict first.** for each store, how many customers will have place 3 or better?

In [ ]:
per_customer["place"] = per_customer.groupby("store_id")["orders"].rank(method="min", ascending=False)
top = per_customer[per_customer["place"] <= 3]
print(top.groupby("store_id").size())

## Unit 3: shift inside each group

**Predict first.** how many rows of `prev` will be empty, and which rows are they?

In [ ]:
store_month["prev"] = store_month.groupby("store_id")["amount"].shift(1)
print(store_month["prev"].isna().sum())
print(store_month[store_month["store_id"] == 3][["month", "amount", "prev"]])

**Predict first.** how many rows will be empty now, and what will sit beside Harbor's April?

In [ ]:
plain = store_month["amount"].shift(1)
first_harbor = store_month.index[store_month["store_id"] == 3][0]
print(plain.isna().sum())
print(plain[first_harbor])

## Unit 4: rolling: a window that slides

**Predict first.** what will the first two values of each list be?

In [ ]:
old_town = store_month[store_month["store_id"] == 1]["amount"]
print(old_town.rolling(3).mean().round(2).tolist())
print(old_town.rolling(3, min_periods=1).mean().round(2).tolist())

**Predict first.** how many empty values will each version have, with and without the `groupby`?

In [ ]:
by_store = store_month.groupby("store_id")["amount"].rolling(3).mean()
plain_roll = store_month["amount"].rolling(3).mean()
print(by_store.isna().sum(), plain_roll.isna().sum())

## Unit 5: Check: two ways to the same numbers

**Predict first.** what will the shares of each store add up to, and does the average that `rolling` gives for Old Town's April equal the mean of February, March and April worked out by hand?

In [ ]:
print(store_month.groupby("store_id")["share"].sum().round(6).tolist())
by_hand = old_town.iloc[1:4].mean()
print(round(old_town.rolling(3).mean().iloc[3], 6) == round(by_hand, 6))

## Finish it

### Exercise 1: Finish it

**Task:** Which product has the biggest share of the sales of its own category? Report that share. `by_product` holds the sales of each product, with its category. Replace the first `None` with the sales of the product's whole category on every row, made with `transform`, then set `answer_1` to the biggest share of a product in its category, a number between 0 and 1.

**Expected:** one decimal number between 0 and 1.

In [ ]:
by_product = sales.groupby(["category", "product_id"])["amount"].sum().reset_index()  # given: one row for each product
by_product["category_total"] = None  # replace None: the sales of the product's category, on every row
answer_1 = None  # then set answer_1 to the largest share: amount divided by category_total

In [ ]:
_ = check("Exercise 1", "answer_1", 1639.0 / 3232.5, traps={0.12863530988716393: "That is a share of all the sales of the chain. The total has to be the total of the product's own category: group first, then transform."})

<details>
<summary>Hint</summary>

The total of a category must be repeated on every row of the category, which is a calculation per group that keeps the rows.

</details>

<details>
<summary>Solution</summary>

```python
by_product = sales.groupby(["category", "product_id"])["amount"].sum().reset_index()
by_product["category_total"] = by_product.groupby("category")["amount"].transform("sum")
answer_1 = (by_product["amount"] / by_product["category_total"]).max()
```

The answer is 0.5070: Chai Latte, product 8, sold 1,639.00 of the 3,232.50 of the tea category, the biggest share any product has in its category. Dividing by the total of the whole chain gives 0.1286, a share of everything, which the check cell flags.

</details>

## On your own

### Exercise 2: On your own

**Task:** What was the biggest rise in the sales of a category from one month to the next? Before you start, say in one sentence what the table you need looks like: one row for each category and month, with the month before beside it. Build it from `sales`, with the sales of each category in each month, and set `answer_2` to the biggest rise in dollars, a decimal number. Write the code in the empty cell below and set `answer_2`.

**Expected:** one decimal number, a rise in dollars.

In [ ]:
_ = check("Exercise 2", "answer_2", 508.5, traps={637.25: "The biggest rise is from the last month of one category to the first month of the next one. Each category needs its own previous row: group before you shift."})

<details>
<summary>Hint</summary>

Group by category and month first. A category's January has no month before it: group before you shift.

</details>

<details>
<summary>Solution</summary>

```python
category_month = sales.groupby(["category", "month"])["amount"].sum().reset_index()
category_month["change"] = category_month["amount"] - category_month.groupby("category")["amount"].shift(1)
answer_2 = category_month["change"].max()
```

The answer is 508.50: food rose from 525.75 in March to 1,034.25 in April, when Harbor opened. Without the `groupby` each category's January is compared with the June of the category above it, and the rise from bakery's June to coffee's January, 637.25, wins; the check cell says so.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague wants the best-selling product of each category by units, and ranks the products inside each category. The list shows the wrong products. Fix the line, then set `answer_3` to the product ids of the best sellers, in the order of the categories (bakery, coffee, food, tea), as a list. `units` has one row for each product.

**Expected:** a list of four product ids.

In [ ]:
units = sales.groupby(["category", "product_id"])["quantity"].sum().reset_index()
units["place"] = units.groupby("category")["quantity"].rank(method="min")  # the colleague's line
answer_3 = None  # fix the line above, then set answer_3 = units[units["place"] == 1]["product_id"].tolist()

In [ ]:
_ = check("Exercise 3", "answer_3", [10, 3, 13, 8], traps={(12, 1, 14, 9): "Place 1 went to the products that sold the fewest units. rank counts from the smallest unless you say otherwise."})

<details>
<summary>Hint</summary>

Print `units` sorted by `place` for one category. Does place 1 hold the product that sold the most or the least?

</details>

<details>
<summary>Solution</summary>

```python
units = sales.groupby(["category", "product_id"])["quantity"].sum().reset_index()
units["place"] = units.groupby("category")["quantity"].rank(method="min", ascending=False)
answer_3 = units[units["place"] == 1]["product_id"].tolist()
```

The answer is `[10, 3, 13, 8]`: Croissant, Latte, Bagel with Cream Cheese and Chai Latte, the best seller of each category. Without `ascending=False` the places count from the smallest, so place 1 went to products 12, 1, 14 and 9, the worst sellers; the check cell says so.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")